In [0]:
%sql

show create table solutions_catalog.pih_schema.consideration_data

In [0]:
%sql

CREATE TABLE solutions_catalog.pih_schema.consideration_data_embeddings (
  id BIGINT GENERATED ALWAYS AS IDENTITY,
  Study_ID BIGINT,
  Brand_Name STRING,
  Advertiser_Name STRING,
  Campaign_ID BIGINT,
  Campaign_Name STRING,
  Publisher_Group STRING,
  Sampled_Exposures BIGINT,
  Engagement_Rate DOUBLE,
  Start_Date TIMESTAMP,
  End_Date TIMESTAMP,
  platform STRING,
  digital_publisher STRING,
  linear_network STRING,
  daypart STRING,
  campaign STRING,
  brand_benchmark DOUBLE,
  brand_index DOUBLE,
  linear_benchmark DOUBLE,
  linear_index DOUBLE,
  digital_benchmark DOUBLE,
  digital_index DOUBLE,
  embeddings ARRAY<FLOAT>)
USING delta
COMMENT 'Created by the file upload UI'
TBLPROPERTIES (
  'delta.minReaderVersion' = '1',
  'delta.minWriterVersion' = '2')



In [0]:
%sql
CREATE TABLE IF NOT EXISTS solutions_catalog.pih_schema.brand_data_embeddings (
    id BIGINT GENERATED ALWAYS AS IDENTITY,
    study_id BIGINT,
    brand_name STRING,
    advertiser_name STRING,
    kpi_metric_name STRING,
    metric_display_name STRING,
    mapping STRING,
    platform STRING,
    control_pct DOUBLE,
    exposed_pct DOUBLE,
    lift_pct DOUBLE,
    update_timestamp TIMESTAMP,
    embeddings ARRAY<FLOAT>
)
USING DELTA;


In [0]:
%pip install mlflow[databricks] --upgrade
dbutils.library.restartPython()


In [0]:
from pyspark.sql.functions import pandas_udf, col, max as spark_max, lit, concat_ws
from pyspark.sql.types import ArrayType, FloatType
import pandas as pd
from mlflow.deployments import get_deploy_client

# Table names
source_table = "solutions_catalog.pih_schema.consideration_data"
target_table = "solutions_catalog.pih_schema.consideration_data_embeddings"

# Step 1: Load max End_Date from embeddings table (as a proxy for update tracking)
try:
    max_timestamp = spark.table(target_table).agg(spark_max("end_date")).collect()[0][0]
except:
    max_timestamp = None

# Step 2: Load only new or changed rows from source
df = spark.table(source_table).select(
    "study_id", "brand_name", "advertiser_name", "campaign_id", "campaign_name",
    "publisher_group", "sampled_exposures", "engagement_rate", "start_date", "end_date",
    "platform", "digital_publisher", "linear_network", "daypart", "campaign",
    "brand_benchmark", "brand_index", "linear_benchmark", "linear_index",
    "digital_benchmark", "digital_index"
)

if max_timestamp:
    df = df.filter(col("end_date") > max_timestamp)

if df.rdd.isEmpty():
    print("✅ No new rows to embed.")
else:
    # Step 3: Create input text column
    df = df.withColumn(
        "embedding_input",
        concat_ws(" ",
            concat_ws(": ", lit("Study ID"), col("study_id").cast("string")),
            concat_ws(": ", lit("Brand"), col("brand_name")),
            concat_ws(": ", lit("Advertiser"), col("advertiser_name")),
            concat_ws(": ", lit("Campaign"), col("campaign_name")),
            concat_ws(": ", lit("Publisher Group"), col("publisher_group")),
            concat_ws(": ", lit("Platform"), col("platform")),
            concat_ws(": ", lit("Digital Publisher"), col("digital_publisher")),
            concat_ws(": ", lit("Linear Network"), col("linear_network")),
            concat_ws(": ", lit("Daypart"), col("daypart")),
            concat_ws(": ", lit("Engagement Rate"), col("engagement_rate").cast("string")),
            concat_ws(": ", lit("Brand Index"), col("brand_index").cast("string")),
            concat_ws(": ", lit("Linear Index"), col("linear_index").cast("string")),
            concat_ws(": ", lit("Digital Index"), col("digital_index").cast("string"))
        )
    )

    # Step 4: Define embedding function using MLflow deployment
    deploy_client = get_deploy_client("databricks")

    @pandas_udf(ArrayType(FloatType()))
    def get_embeddings(texts: pd.Series) -> pd.Series:
        clean_texts = texts.fillna("").astype(str).replace("nan", "", regex=False)
        clean_texts = clean_texts.apply(lambda x: x.strip() if x.strip() != "" else "[EMPTY TEXT]")

        batch_size = 150
        inputs = clean_texts.tolist()
        all_embeddings = []

        for i in range(0, len(inputs), batch_size):
            chunk = inputs[i:i + batch_size]
            result = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": chunk})

            if isinstance(result["data"], list):
                if all(isinstance(item, list) for item in result["data"]):
                    all_embeddings.extend(result["data"])
                elif all(isinstance(item, dict) and "embedding" in item for item in result["data"]):
                    all_embeddings.extend([item["embedding"] for item in result["data"]])
                else:
                    raise ValueError("Unexpected format in 'data' field of response.")
            else:
                raise ValueError("Expected 'data' to be a list.")

        return pd.Series(all_embeddings)

    # Step 5: Generate embeddings and write to target table
    df_with_embeddings = df.withColumn("embeddings", get_embeddings("embedding_input")).drop("embedding_input")

    df_with_embeddings.write.mode("append").format("delta").saveAsTable(target_table)

    print(f"✅ Appended {df_with_embeddings.count()} new rows with embeddings to {target_table}")


In [0]:
%sql

select * from solutions_catalog.pih_schema.consideration_data_embeddings


In [0]:
%sql

select distinct platform from solutions_catalog.pih_schema.brand_data_embeddings